# 10 — Backtest and Performance

Този notebook събира наличните backtest/performance reports и ги показва като таблици и графики.


In [ ]:
from pathlib import Path
from itertools import combinations
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 160)

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"
REPORTS_DIR = PROJECT_ROOT / "reports"
MODELS_DIR = PROJECT_ROOT / "models"


def read_csv_safe(path, **kwargs):
    path = Path(path)
    if not path.exists():
        print(f"Missing file: {path}")
        return pd.DataFrame()
    return pd.read_csv(path, **kwargs)


def number_columns(df: pd.DataFrame) -> list[str]:
    return [col for col in ["n1", "n2", "n3", "n4", "n5", "n6"] if col in df.columns]


def parse_numbers_text(value) -> list[int]:
    if pd.isna(value):
        return []
    parts = str(value).replace(";", ",").replace("|", ",").split(",")
    nums = []
    for part in parts:
        part = part.strip()
        if part.isdigit():
            nums.append(int(part))
    return nums


def extract_ticket_numbers(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return df
    if len(number_columns(df)) == 6:
        return df.copy()
    text_col = next((c for c in df.columns if "numbers" in c.lower() or "числа" in c.lower()), None)
    if text_col is None:
        return df.copy()
    out = df.copy()
    parsed = out[text_col].apply(parse_numbers_text)
    for i in range(6):
        out[f"n{i+1}"] = parsed.apply(lambda nums: nums[i] if len(nums) > i else np.nan)
    return out


def show_latest_draw(df: pd.DataFrame) -> None:
    if df.empty:
        print("No draw data loaded.")
        return
    latest = df.tail(1).iloc[0]
    nums = [int(latest[col]) for col in number_columns(df)]
    print("Rows:", len(df))
    print("Latest date:", latest.get("date", "n/a"))
    print("Draw number:", latest.get("draw_number", latest.get("draw_no", "n/a")))
    print("Numbers:", nums)


def file_status(paths):
    rows = []
    for path in paths:
        path = Path(path)
        rows.append({
            "file": str(path.relative_to(PROJECT_ROOT)) if str(path).startswith(str(PROJECT_ROOT)) else str(path),
            "exists": path.exists(),
            "size_kb": round(path.stat().st_size / 1024, 2) if path.exists() else 0,
        })
    return pd.DataFrame(rows)


draws = read_csv_safe(DATA_DIR / "historical_draws.csv")
normalized = read_csv_safe(DATA_DIR / "v40_normalized_draw_events.csv")
canonical = read_csv_safe(DATA_DIR / "v41_canonical_draw_events.csv")
show_latest_draw(draws)


## Backtest files inventory


In [ ]:
backtest_files = sorted(set(
    list(REPORTS_DIR.glob("*backtest*.csv"))
    + list(REPORTS_DIR.glob("*performance*.csv"))
    + list(REPORTS_DIR.glob("*result*.csv"))
))
pd.DataFrame({
    "file": [str(path.relative_to(PROJECT_ROOT)) for path in backtest_files],
    "size_kb": [round(path.stat().st_size / 1024, 2) for path in backtest_files],
}).head(100)


## Latest ticket pack result


In [ ]:
latest_result = read_csv_safe(REPORTS_DIR / "v73_latest_ticket_pack_result.csv")
latest_result


## Ticket pack performance history


In [ ]:
history = read_csv_safe(REPORTS_DIR / "v73_ticket_pack_performance_history.csv")
display(history.head())
if not history.empty and "best_hit_count" in history.columns:
    ax = history["best_hit_count"].plot(figsize=(12, 4), title="Performance history: best_hit_count")
    ax.set_xlabel("Row")
    ax.set_ylabel("best_hit_count")
    plt.show()


## Strategy backtest


In [ ]:
strategy = read_csv_safe(REPORTS_DIR / "v92_strategy_backtest_results.csv")
display(strategy.head(20))
if not strategy.empty and "average_best_hits" in strategy.columns:
    label_col = "strategy_label" if "strategy_label" in strategy.columns else strategy.columns[0]
    ax = strategy.set_index(label_col)["average_best_hits"].plot(kind="bar", figsize=(14, 5), title="Strategy backtest: average_best_hits")
    ax.set_xlabel("Strategy")
    ax.set_ylabel("average_best_hits")
    plt.show()
